# 02 Brazil: baselines without climate (Steps 3-6)

**Task.** For each municipality-year we hide the weekly counts and keep only the annual total. A model predicts the 52/53 weekly *shares*; shares x total, rounded with the largest-remainder rule, gives integer weekly counts that add up exactly to the total.

**Validation.** Leave-one-year-out (LOYO) over 2010-2023 (14 folds, each holds out one year for all municipalities). Everything estimated from data is recomputed inside each fold from training years only. **2024 is never loaded.**

| ID | Model |
|---|---|
| B0 | uniform shares |
| B1 | municipality's historical mean weekly shares (LOYO) |
| B1p | same, years before the target year only |
| B4 | Dirichlet-multinomial with the base paper's structure (offset + national/regional/municipal week deviations, smoothness penalty, one concentration per region) |
| B4s | B4 + annual-size terms - **primary comparator for climate** |

Results are precomputed; set `RERUN = True` to refit (weekly, 60 municipalities, main models: about 5 min on 4 CPUs).

In [ ]:
# ---- Setup: works in Google Colab and locally (VS Code / Jupyter) ----
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/aminah9945/dengue-climate-reconstruction"
BRANCH = "main"          # use "official-run" if the pull request is not merged yet
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = Path("/content/dengue-climate-reconstruction")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[all]"], check=True)
    # Optional: keep the downloaded data on Google Drive so it survives the session.
    USE_DRIVE = False
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ.setdefault("DATA_DIR", "/content/drive/MyDrive/dengue_recon_data")
else:
    # Local: this notebook lives in <repo>/notebooks/
    REPO = Path(os.environ.get("DENGUE_ROOT", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())).resolve()

os.environ.setdefault("DENGUE_ROOT", str(REPO))
os.environ.setdefault("DATA_DIR", str(REPO / "data"))
os.environ.setdefault("RESULTS_DIR", str(REPO / "results"))
os.environ.setdefault("XLA_FLAGS", "--xla_force_host_platform_device_count=1")
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))      # makes the package importable without a kernel restart

from dengue_recon import config as C
C.ensure_dirs()
print("repo       :", C.ROOT)
print("DATA_DIR   :", C.DATA_DIR, "| processed data present:", (C.DATA_DIR / "dengue_weekly.parquet").exists())
print("RESULTS_DIR:", C.RESULTS_DIR)

In [ ]:
import json
import numpy as np, pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.width", 200, "display.max_columns", 30)


def show(path, width=900):
    p = Path(path)
    p = p if p.is_absolute() else C.FIG_DIR / p
    display(Image(str(p), width=width)) if p.exists() else print("missing figure:", p)


def have_data():
    ok = (C.DATA_DIR / "dengue_weekly.parquet").exists()
    if not ok:
        print("Processed data not found in DATA_DIR - run notebook 00 first (or set DATA_DIR).")
    return ok

In [ ]:
RERUN = False
if RERUN and have_data():
    from dengue_recon.experiments import brazil_loyo
    brazil_loyo.main("weekly", 60, models=["B4", "B4s"], tag="weekly_60_noclimate")
sub = pd.read_csv(C.RESULTS_DIR / "subset60.csv")
print(sub.groupby("region").size())

### Live demo: B0 and B1 for one fold (no fitting needed; needs the data)

In [ ]:
if have_data():
    from dengue_recon.baselines import seasonal_shares, uniform_shares
    from dengue_recon.data import panel as BD
    from dengue_recon.evaluation.cv import loyo_folds, row_masks
    from dengue_recon.evaluation.metrics import jsd
    from dengue_recon.features.seasonal import offsets
    d, clim, muni = BD.load_tables()
    P = BD.build_panel(d, clim, sub.geocode.values)
    train_ok, eval_ok = row_masks(P)
    t, y, tm = next(loyo_folds(P, train_ok, years=[2019]))
    h, fb, prof = offsets(P, tm, weekly=True)
    rows = np.where(eval_ok[:, t])[0]; n = P.nweeks[t]
    j0 = [jsd(P.Yw[m, t, :n] / P.N[m, t], uniform_shares(n)) for m in rows]
    j1 = [jsd(P.Yw[m, t, :n] / P.N[m, t], seasonal_shares(prof[m], n)) for m in rows]
    print(f"{y}, {len(rows)} municipality-years: JSD uniform {np.mean(j0):.3f}, seasonal profile {np.mean(j1):.3f}")

### Main table (all LOYO folds)

In [ ]:
cols = ["model", "subset", "n", "jsd_mean", "jsd_median", "peak_err_mean", "emd_mean", "mae_share_mean", "cov80_mean", "cov95_mean", "crps_mean", "jsd_Nweighted"]
for res in ["monthly", "weekly"]:
    t_ = pd.read_csv(C.BRAZIL_DIR / f"table_main_{res}.csv")
    display(Markdown(f"**{res}** - {t_.note.iloc[0]}"))
    display(t_[(t_.subset == "all") & t_.model.isin(["B0", "B1p", "B1", "B4", "B4s", "B4s-eps1e-4"])][[c for c in cols if c in t_]].round(4))

### Paired differences between baselines (negative = first model better)

In [ ]:
p = pd.read_csv(C.BRAZIL_DIR / "table_paired_weekly.csv")
p[(p.subset == "all") & (p.metric == "jsd") & p.model_a.isin(["B4s", "B4", "B1", "B4s-eps1e-4"])][
    ["model_a", "model_b", "n", "mean_diff", "rel_change", "ci_muni_lo", "ci_muni_hi", "ci_year_lo", "ci_year_hi", "n_years_a_better"]].round(5)